# Z potential, Langevin dynamics: committor from many short trajectories

The 2D Z potential of the staple runs (toytis `sjoelbak_angle`, Lx = 0.3, 45° ridge) with **Langevin dynamics**: γ = 2, kT = 0.05, m = 1, dt = 0.002 (the staple run `sim_istarz_2603`; A: y ≤ 0.05, B: y ≥ 0.85, y = order parameter). Same notebook as `zpot_short_trajectories.ipynb` (overdamped, γ = 10); what changes for Langevin:

- **Phase space.** The committor depends on the velocities too. The short trajectories are real Langevin trajectories with velocities, so the fit uses the CVs (y, x, vy, vx).
- **Start velocities.** The staple run stores positions only (`v_ord = False`), so a start frame provides a position and its velocities are drawn from the Maxwell distribution. The fitted q is then the committor of a configuration with equilibrium velocities.
- **No exact committor.** The grid solution only exists for overdamped dynamics. The reference is shooting from the start positions with Maxwell velocities (same engine as the short trajectories, run to absorption), and the first try.
- **Longer trajectories.** The velocity relaxes over 1/γ = 250 steps and the dynamics is faster than at γ = 10, so `L` is larger.

**Part 1** (original `optimalrcs-cupy`): take a saved first-try committor (any `committor.npz` with the frames and their q), pick start frames evenly over bins of its natural coordinate q_n (or of q), run many short Langevin trajectories from them, and fit NPNEq (`CommittorNE`) on those. The new trajectories are plain dynamics (no path selection, no weights), so the fit is unbiased whatever the start frames are; the bins only decide where the data are. The result can be saved in the same format and used as the next first try.

**Part 2** (Elias' TIS version of the library, restart the kernel first): read the Langevin staple paths of `sim_istarz_2603` and fit the committor on them (CVs y, x and the per-frame changes dy, dx as velocities, lagged fit); it saves the `committor.npz` that part 1 can start from.

## Part 1: short trajectories from a first-try committor

### Imports (original optimalrcs-cupy)

In [ ]:
import os
import sys
import time

import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

plt.rc('font', size=14)          # controls default text sizes
plt.rc('axes', titlesize=16)     # fontsize of the axes title
plt.rc('axes', labelsize=16)     # fontsize of the x and y labels
plt.rc('xtick', labelsize=14)    # fontsize of the tick labels
plt.rc('ytick', labelsize=14)    # fontsize of the tick labels
plt.rc('legend', fontsize=12)    # legend fontsize

ORIG_LIB = '/home/staff7/fbsskr/Notebooks/optimalrcs-cupy/'      # the original optimalrcs-cupy
sys.path.append(ORIG_LIB)
import optimalrcs, metrics, plots

FIG = 'figures/nb_zpot_short_traj_langevin'                               # figures and data of this notebook
os.makedirs(FIG, exist_ok=True)

def show(call, name):
    """Run a plotting call of the library (which calls plt.show itself), save the figure, then show it."""
    plt_show = plt.show
    plt.show = lambda *a, **k: None
    try:
        call()
    finally:
        plt.show = plt_show
    plt.gcf().savefig(f'{FIG}/{name}.png', dpi=100, bbox_inches='tight')
    plt.show()

### The Z potential

`force(y, x)` is the force of toytis' `RectangularGridWithAngledBarrierPotential(Lx=0.3, angle=45)` term by term (walls, the tilted cosine ridge, the cosine wells of A and B, the slope between them and gravity), on GPU arrays. `potential(y, x)` is its effective potential (the class' `potential` plus gravity × y, so that force = −∇V). If toytis can be imported, the force is checked against the class.

In [ ]:
kT, gamma_md, mass, dt = 0.05, 2.0, 1.0, 0.002   # Langevin dynamics of sim_istarz_2603
lambda_A, lambda_B = 0.05, 0.85            # A: y <= lambda_A, B: y >= lambda_B

# parameters of the potential (toytis defaults, Lx = 0.3, angle = 45 degrees)
Lx, Ly, k_wall = 0.3, 0.9, 2000.0
A_ridge, L_ridge = 0.3, 0.2                # ridge height and length scale (half-width L/4)
slope_r = -np.tan(np.radians(45.0))        # the ridge line y = slope_r * x + b passes through (x, y) = (Lx/2, 0.45)
b_ridge = 0.45 - slope_r * Lx / 2
wells = [(0.025, -0.15, 0.2), (0.875, -0.15, 0.2)]   # (centre y, amplitude, length scale) of the wells of A and B
slope_start, slope_end, slope_strength = 0.1, 0.825, 0.2
gravity = 0.2

def force(y, x, xp=cp):
    """Force (F_y, F_x) of the Z potential at arrays (y, x)."""
    fx = xp.where(x < 0, -k_wall * x, xp.where(x > Lx, -k_wall * (x - Lx), 0.0))        # walls
    fy = xp.where(y < 0, -k_wall * y, xp.where(y > Ly, -k_wall * (y - Ly), 0.0))
    norm = np.sqrt(slope_r ** 2 + 1)                                                     # ridge
    D = xp.abs(slope_r * x - y + b_ridge) / norm
    dV_dD = xp.where(D < L_ridge / 4, -2 * np.pi * A_ridge / L_ridge * xp.sin(2 * np.pi * D / L_ridge), 0.0)
    side = xp.sign(y - slope_r * x - b_ridge)
    fx = fx + side * dV_dD * slope_r / norm
    fy = fy - side * dV_dD / norm
    for yc, amp, ell in wells:                                                           # wells of A and B
        Dw = xp.abs(y - yc)
        fy = fy + xp.where(Dw < ell / 4, amp * 2 * np.pi / ell * xp.sin(2 * np.pi * Dw / ell) * xp.sign(y - yc), 0.0)
    inside = (y >= slope_start) & (y <= slope_end)                                       # slope and gravity
    fy = fy + xp.where(inside, -slope_strength / (slope_end - slope_start), 0.0) - gravity
    return fy, fx

def potential(y, x, xp=np):
    """Effective potential of the Z potential (force = -grad V)."""
    v = xp.where(x < 0, 0.5 * k_wall * x ** 2, xp.where(x > Lx, 0.5 * k_wall * (x - Lx) ** 2, 0.0))
    v = v + xp.where(y < 0, 0.5 * k_wall * y ** 2, xp.where(y > Ly, 0.5 * k_wall * (y - Ly) ** 2, 0.0))
    D = xp.abs(slope_r * x - y + b_ridge) / np.sqrt(slope_r ** 2 + 1)
    v = v + xp.where(D < L_ridge / 4, A_ridge * xp.cos(2 * np.pi * D / L_ridge), 0.0)
    for yc, amp, ell in wells:
        Dw = xp.abs(y - yc)
        v = v + xp.where(Dw < ell / 4, amp * xp.cos(2 * np.pi * Dw / ell), 0.0)
    v = v + xp.where(y < slope_start, 0.0, xp.where(y <= slope_end, slope_strength * (y - slope_start) / (slope_end - slope_start), slope_strength))
    return v + gravity * y

# check against the toytis class, if it can be imported
TOYTIS = '/home/staff7/fbsskr/Notebooks/Students/elias/toytis'
try:
    sys.path.insert(0, TOYTIS)
    from potentials.sjoelbak_angle import RectangularGridWithAngledBarrierPotential
    pot = RectangularGridWithAngledBarrierPotential(Lx=0.3, angle=45)
    pts = np.random.default_rng(1).uniform([-0.02, -0.02], [0.92, 0.32], size=(300, 2))
    f_here = np.column_stack([f.get() for f in force(cp.asarray(pts[:, 0]), cp.asarray(pts[:, 1]))])
    f_class = np.array([pot.potential_and_force((p, np.zeros(2)))[1] for p in pts])
    v_here = potential(pts[:, 0], pts[:, 1])
    v_class = np.array([pot.effective_potential((p, np.zeros(2))) for p in pts])
    print('force and potential agree with the toytis class:', np.allclose(f_here, f_class, atol=1e-9), np.allclose(v_here, v_class, atol=1e-9))
except ImportError:
    print('toytis not found: force and potential not checked against the class')

gy, gx = np.linspace(-0.02, 0.92, 189), np.linspace(-0.02, 0.32, 69)
V = potential(*np.meshgrid(gy, gx, indexing='ij'))
V_levels = np.arange(0, 12.5, 1.0)
def draw_potential(ax):
    ax.contour(gy, gx, ((V - V.min()) / kT).T, levels=V_levels, colors='0.6', linewidths=0.5)
    for lam in (lambda_A, lambda_B):
        ax.axvline(lam, color='k', lw=1)
    ax.set(xlabel='y (order parameter)', ylabel='x', xlim=(-0.02, 0.92), ylim=(-0.02, 0.32))

### The Langevin dynamics

The toytis Langevin update (`vvengine`, as `toy2d.Integrator` with `high_friction=False`): with c0 = exp(−γ dt), c1 = (1 − c0) / (γ dt), c2 = (1 − c1) / (γ dt), the new position is r + c1 dt v + c2 dt² F / m + noise, and the new velocity is c0 v + (c1 − c2) dt F / m + noise + c2 dt F_new / m, with F_new the force at the new position. The two noise terms of a step are correlated Gaussians (covariance matrix as in `toy2d.Integrator`). Absorption is checked every step.

`step` advances arrays of walkers by one step and reuses the force of the new position for the next step. The check draws walkers in the well of A, thermalises them and compares ⟨v²⟩ with kT/m (equipartition).

In [ ]:
c0 = np.exp(-gamma_md * dt)
c1 = (1 - c0) / (gamma_md * dt)
c2 = (1 - c1) / (gamma_md * dt)
a1, a2 = c1 * dt, c2 * dt ** 2 / mass
b1, b2 = (c1 - c2) * dt / mass, c2 * dt / mass
s_r2 = (dt * kT / (gamma_md * mass)) * (2. - (3. - 4. * c0 + c0 ** 2) / (gamma_md * dt))   # variance of the position noise
s_v2 = (1. - c0 ** 2) * kT / mass                                                          # variance of the velocity noise
cov = (kT / (mass * gamma_md)) * (1. - c0) ** 2
l11 = np.sqrt(s_r2); l21 = cov / l11; l22 = np.sqrt(s_v2 - l21 ** 2)

def step(y, x, vy, vx, fy, fx):
    """One Langevin step of arrays of walkers; fy, fx: forces at (y, x). Returns the new state and the new forces."""
    n = y.shape[0]
    z1y, z2y, z1x, z2x = (cp.random.standard_normal(n) for _ in range(4))
    y_new = y + a1 * vy + a2 * fy + l11 * z1y
    x_new = x + a1 * vx + a2 * fx + l11 * z1x
    vy2 = c0 * vy + b1 * fy + l21 * z1y + l22 * z2y
    vx2 = c0 * vx + b1 * fx + l21 * z1x + l22 * z2x
    fy_n, fx_n = force(y_new, x_new)
    return y_new, x_new, vy2 + b2 * fy_n, vx2 + b2 * fx_n, fy_n, fx_n

def maxwell(n):
    return np.sqrt(kT / mass) * cp.random.standard_normal(n), np.sqrt(kT / mass) * cp.random.standard_normal(n)

# equipartition check: 5000 walkers started in the well of A, 3000 steps
cp.random.seed(0)
y_c, x_c = cp.full(5000, 0.05), cp.full(5000, 0.15)
vy_c, vx_c = maxwell(5000)
fy_c, fx_c = force(y_c, x_c)
for _ in range(3000):
    y_c, x_c, vy_c, vx_c, fy_c, fx_c = step(y_c, x_c, vy_c, vx_c, fy_c, fx_c)
print(f'<vy^2> / (kT/m) = {float(cp.mean(vy_c ** 2)) / (kT / mass):.3f}, <vx^2> / (kT/m) = {float(cp.mean(vx_c ** 2)) / (kT / mass):.3f}  (1 expected)')
del y_c, x_c, vy_c, vx_c, fy_c, fx_c

### The first-try committor

Any `committor.npz` with the frames and their committor: `X_all` (columns y, x, and any further CVs, which are not used here), `r` (or `q_fit`), and for q_n also `i_traj`, `t_traj` (and `path_weights` if the frames are path data). Part 2 saves such a file.

`BIN_ON = 'qn'`: start frames evenly over bins of the natural coordinate q_n of the first try (`plots.transform_q2qn`): equal steps in q_n are steps of equal difficulty for the dynamics, so the starts are spread over the transition region instead of piling up where the paths spend their time. `BIN_ON = 'q'`: bins of q itself.

Only the positions of the first-try frames are used as starts: the frames carry no velocities (finite-difference velocities of a path are not phase-space velocities), so the start velocities are drawn from the Maxwell distribution.

In [ ]:
FIRST_TRY_NPZ = f'{FIG}/committor_staple_langevin_p1500.npz'      # written by part 2
BIN_ON = 'qn'              # 'qn' or 'q'
N_BINS = 30

d = np.load(FIRST_TRY_NPZ)
q_first = (d['r'] if 'r' in d.files else d['q_fit']).astype(float)
y_first, x_first = d['X_all'][:, 0].astype(float), d['X_all'][:, 1].astype(float)
free_first = (y_first > lambda_A) & (y_first < lambda_B)
print(f'{FIRST_TRY_NPZ}: {len(q_first):,} frames, {free_first.sum():,} outside A and B')

if BIN_ON == 'qn':
    w_first = cp.asarray(d['path_weights']) if 'path_weights' in d.files else None
    qn_first = plots.transform_q2qn(cp.asarray(q_first), i_traj=cp.asarray(d['i_traj']), t_traj=cp.asarray(d['t_traj']),
                                    w_traj=w_first, dt_sim=dt)
    coord = qn_first / qn_first.max()
else:
    coord = q_first
bin_first = np.clip((coord * N_BINS).astype(int), 0, N_BINS - 1)
n_per_bin = np.bincount(bin_first[free_first], minlength=N_BINS)
print(f'frames per bin of {BIN_ON}:', n_per_bin)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))
show_frames = np.random.default_rng(0).choice(np.flatnonzero(free_first), size=min(200_000, free_first.sum()), replace=False)
sc = ax1.scatter(y_first[show_frames], x_first[show_frames], c=q_first[show_frames], s=1, cmap='coolwarm', vmin=0, vmax=1)
draw_potential(ax1)
fig.colorbar(sc, ax=ax1, label='first-try q')
ax1.set_title('first-try committor at its frames')
ax2.bar(np.arange(N_BINS), n_per_bin)
ax2.set(yscale='log', xlabel=f'bin of {BIN_ON}', ylabel='frames', title='frames per bin (start frames are drawn evenly over the bins)')
fig.tight_layout()
fig.savefig(f'{FIG}/first_try.png', dpi=100, bbox_inches='tight')
plt.show()

### Many short trajectories

`N_TRAJ` start frames, `N_TRAJ / N_BINS` per bin (fewer if a bin has fewer frames), one Langevin trajectory of at most `L` steps from each, with a Maxwell start velocity, stopped when it reaches A or B. All trajectories run at once on the GPU; a stopped trajectory is frozen and its later frames are not used. The velocities of the frames are kept (CVs of the fit).

The default `L = 1000` steps (2 time units, 8 velocity relaxation times) is the time a trajectory gets to reach A or B or to forget its start; the printed fraction that reaches A or B tells whether it is long enough.

In [ ]:
N_TRAJ = 20_000
L = 1000                   # steps per trajectory
SEED = 0

rng = np.random.default_rng(SEED)
start = np.concatenate([rng.choice(np.flatnonzero(free_first & (bin_first == b)), size=min(N_TRAJ // N_BINS, n_per_bin[b]), replace=False)
                        for b in range(N_BINS)])
n_traj = len(start)

cp.random.seed(SEED)
y_now, x_now = cp.asarray(y_first[start]), cp.asarray(x_first[start])
vy_now, vx_now = maxwell(n_traj)
fy_now, fx_now = force(y_now, x_now)
Y_frames, X_frames = cp.zeros((L + 1, n_traj)), cp.zeros((L + 1, n_traj))      # [step, trajectory]
VY_frames, VX_frames = cp.zeros((L + 1, n_traj)), cp.zeros((L + 1, n_traj))
Y_frames[0], X_frames[0], VY_frames[0], VX_frames[0] = y_now, x_now, vy_now, vx_now
last = cp.full(n_traj, L)                       # last frame of each trajectory
running = cp.ones(n_traj, bool)
t0 = time.time()
for it in range(1, L + 1):
    y_new, x_new, vy_new, vx_new, fy_new, fx_new = step(y_now, x_now, vy_now, vx_now, fy_now, fx_now)
    y_now, x_now = cp.where(running, y_new, y_now), cp.where(running, x_new, x_now)              # stopped: frozen
    vy_now, vx_now = cp.where(running, vy_new, vy_now), cp.where(running, vx_new, vx_now)
    fy_now, fx_now = cp.where(running, fy_new, fy_now), cp.where(running, fx_new, fx_now)
    Y_frames[it], X_frames[it], VY_frames[it], VX_frames[it] = y_now, x_now, vy_now, vx_now
    absorbed = running & ((y_now <= lambda_A) | (y_now >= lambda_B))
    last = cp.where(absorbed, it, last)
    running = running & ~absorbed
last = last.get()
print(f'{n_traj} trajectories of up to {L} steps in {time.time() - t0:.1f} s; '
      f'{np.mean(last < L):.0%} reached A or B')

# one segment per trajectory: frames 0 .. last
keep = np.arange(L + 1)[None, :] <= last[:, None]                  # [trajectory, step]
y, x = Y_frames.get().T[keep], X_frames.get().T[keep]
vy, vx = VY_frames.get().T[keep], VX_frames.get().T[keep]
i_traj = np.repeat(np.arange(n_traj), last + 1)
t_traj = np.concatenate([np.arange(n + 1) for n in last]).astype(float)
first_frame = np.r_[0, np.cumsum(last + 1)[:-1]]                  # index of each start frame
del Y_frames, X_frames, VY_frames, VX_frames
print(f'{len(y):,} frames; {np.sum(y <= lambda_A)} in A, {np.sum(y >= lambda_B)} in B')
np.savez(f'{FIG}/short_traj_{BIN_ON}_n{n_traj}_L{L}.npz', y=y, x=x, vy=vy, vx=vx, i_traj=i_traj, t_traj=t_traj, start=start)

fig, ax = plt.subplots(figsize=(11, 5))
draw_potential(ax)
ax.scatter(y[first_frame], x[first_frame], s=2, c='C0', label='start frames')
for j in np.random.default_rng(1).choice(n_traj, 30, replace=False):
    ax.plot(y[i_traj == j], x[i_traj == j], lw=0.6)
ax.legend(loc='upper left')
ax.set_title(f'{n_traj} start frames over {N_BINS} bins of {BIN_ON}, and 30 of the trajectories')
fig.savefig(f'{FIG}/trajectories.png', dpi=100, bbox_inches='tight')
plt.show()

### NPNEq fit on the short trajectories

`CommittorNE` with one segment per trajectory. The trajectories that reach A or B are treated as staying there (`set_fixed_traj_length_trap`; this only affects the Z_q plots, not the fit). CVs: y, x, vy and vx, one drawn at random every iteration (`CVS` selects them; `('y', 'x')` fits the committor of the positions only, which is not Markov for Langevin dynamics and should do worse). A warm-up with a very wide envelope, then the usual envelope 0.01. Lag 1: the trajectories carry the velocities, so the phase-space dynamics is Markov and the standard NPNEq step applies.

In [ ]:
CVS = ('y', 'x', 'vy', 'vx')

boundary0, boundary1 = y <= lambda_A, y >= lambda_B

q = optimalrcs.CommittorNE(boundary0=boundary0, boundary1=boundary1, i_traj=i_traj, t_traj=t_traj)
q.set_fixed_traj_length_trap(boundary0 | boundary1, 2 ** 16)

cv_all = dict(y=y, x=x, vy=vy, vx=vx)
cvs = [cp.asarray(cv_all[c]) for c in CVS]
def comp_y():
    return cvs[np.random.randint(len(cvs))]

metrics_print = ('iter', 'delta_r2', 'max_sd_zq', 'delta_x', 'time_elapsed')
np.random.seed(0)
cp.random.seed(0)

optimalrcs.envelope_scale = 1e7          # warm-up
q.fit_transform(comp_y, gamma=0, ny=6, max_iter=1000, print_step=500, metrics_print=metrics_print,
                delta_r2_max_change_allowed=10000)
optimalrcs.envelope_scale = 0.01         # main fit
q.fit_transform(comp_y, gamma=0, ny=6, max_iter=20000, print_step=1000, metrics_print=metrics_print)

In [ ]:
show(lambda: q.plots_metrics(['delta_r2', 'max_sd_zq', 'delta_x']), 'fit_metrics')
show(lambda: q.plots_feps(delta_t_sim=dt), 'fit_feps')
show(lambda: q.plots_obs_pred(), 'fit_obs_pred')
show(lambda: q.plots_obs_pred(log_scale=True), 'fit_obs_pred_log')

### Comparison with shooting and the first try

There is no exact committor for Langevin dynamics here. The reference is **shooting**: from four plateau positions beyond the ridge, `N_SHOT` Langevin trajectories with Maxwell velocities, run until A or B (the same engine as the short trajectories, so this checks the fit, not the integrator). The fit and the first try are averaged over the frames within 0.01 of each position; the frames of the short trajectories have the velocity distribution of the dynamics from the start frames, the first-try frames that of the paths, so small differences are expected.

Maps: the fit and the first try averaged per (y, x) bin (bins with at least 20 frames), and their difference with its RMS over the bins where the first try is between 0.05 and 0.95. Without an exact committor the difference measures how far the two fits are from each other, not the error of either.

In [ ]:
r = q.r_traj.get()
free = ~(boundary0 | boundary1)
ey, ex = np.linspace(lambda_A, lambda_B, 81), np.linspace(-0.01, 0.31, 33)

def binned_mean(yy, xx, values, mask, weights=None):
    w = np.ones_like(values) if weights is None else weights
    n = np.histogram2d(yy[mask], xx[mask], bins=(ey, ex))[0]
    sw = np.histogram2d(yy[mask], xx[mask], bins=(ey, ex), weights=w[mask])[0]
    s = np.histogram2d(yy[mask], xx[mask], bins=(ey, ex), weights=(w * values)[mask])[0]
    return np.where(n >= 20, s / np.maximum(sw, 1e-300), np.nan)

Q_fit = binned_mean(y, x, r, free)
Q_first = binned_mean(y_first, x_first, q_first, free_first, d['path_weights'].astype(float) if 'path_weights' in d.files else None)
transition = (Q_first > 0.05) & (Q_first < 0.95)
rms = np.sqrt(np.nanmean((Q_fit - Q_first)[transition] ** 2))
print(f'RMS(new fit - first try) per (y, x) bin, 0.05 < first try < 0.95: {rms:.4f}')

# shooting at four plateau positions, Maxwell velocities, until A or B
N_SHOT, MAX_SHOOT_STEPS = 1000, 20000
points = [(0.50, 0.25), (0.55, 0.10), (0.62, 0.15), (0.70, 0.15)]
cp.random.seed(5)
py = cp.asarray(np.repeat([p[0] for p in points], N_SHOT).astype(float))
px = cp.asarray(np.repeat([p[1] for p in points], N_SHOT).astype(float))
sy, sx = py.copy(), px.copy()
svy, svx = maxwell(len(sy))
sfy, sfx = force(sy, sx)
outcome = cp.full(len(sy), -1.0)
t0 = time.time()
for it in range(MAX_SHOOT_STEPS):
    sy, sx, svy, svx, sfy, sfx = step(sy, sx, svy, svx, sfy, sfx)
    outcome = cp.where((outcome < 0) & (sy <= lambda_A), 0.0, outcome)
    outcome = cp.where((outcome < 0) & (sy >= lambda_B), 1.0, outcome)
    if it % 200 == 199 and bool(cp.all(outcome >= 0)):
        break
outcome = outcome.get().reshape(len(points), N_SHOT)
print(f'shooting: {len(points)} x {N_SHOT} trajectories in {time.time() - t0:.0f} s, {np.mean(outcome < 0):.1%} unresolved after {it + 1} steps')
print('point (y, x)    shooting +- se   new fit   first try')
for k, (py_, px_) in enumerate(points):
    done = outcome[k] >= 0
    q_shoot = outcome[k][done].mean()
    se = np.sqrt(q_shoot * (1 - q_shoot) / done.sum())
    near = free & ((y - py_) ** 2 + (x - px_) ** 2 < 0.01 ** 2)
    near_first = free_first & ((y_first - py_) ** 2 + (x_first - px_) ** 2 < 0.01 ** 2)
    print(f'({py_:.2f}, {px_:.2f})     {q_shoot:.3f} +- {se:.3f}    {r[near].mean():.3f}     {q_first[near_first].mean():.3f}')

panels = [(Q_fit, 'new fit', 'coolwarm', 0, 1), (Q_first, 'first try', 'coolwarm', 0, 1),
          (Q_fit - Q_first, f'new fit - first try, RMS {rms:.4f}', 'PuOr', -0.15, 0.15)]
fig, axes = plt.subplots(1, len(panels), figsize=(7 * len(panels), 4.5), squeeze=False)
for ax, (M, title, cmap, vmin, vmax) in zip(axes[0], panels):
    im = ax.pcolormesh(ey, ex, M.T, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.contour(gy, gx, ((V - V.min()) / kT).T, levels=V_levels, colors='0.6', linewidths=0.5)
    ax.plot([p[0] for p in points], [p[1] for p in points], 'k+', ms=10)
    ax.set(title=title, xlabel='y', ylabel='x', xlim=(lambda_A, lambda_B), ylim=(-0.01, 0.31))
    fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(f'{FIG}/maps.png', dpi=100, bbox_inches='tight')
plt.show()

# profile along y: mean over x of the binned maps
mid = (ey[1:] + ey[:-1]) / 2
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(mid, np.nanmean(Q_fit, axis=1), label='new fit')
ax.plot(mid, np.nanmean(Q_first, axis=1), label='first try')
ax.set(xlabel='y (order parameter)', ylabel='q (mean over x bins)')
ax.legend()
ax.grid(alpha=0.3)
fig.savefig(f'{FIG}/profile.png', dpi=100, bbox_inches='tight')
plt.show()

### Save

In the format the first-try cell reads (`X_all` = y, x, vy, vx, `r`, `i_traj`, `t_traj`), so the next round can start from this fit (set `FIRST_TRY_NPZ` to it; only the positions are used as starts).

In [ ]:
OUT_NPZ = f'{FIG}/committor_short_traj_{BIN_ON}_n{n_traj}_L{L}.npz'
np.savez(OUT_NPZ, r=r, X_all=np.column_stack([y, x, vy, vx]), i_traj=i_traj, t_traj=t_traj,
         boundary0=boundary0, boundary1=boundary1, start=start)
print('saved', OUT_NPZ)

## Part 2: Langevin staple paths and their committor (Elias' TIS version)

This part needs the modified library (`optimalrcs-cupy` of the `partial_tis` branch: path weights, independence groups, stopped Z_q, lazy basis) and `examples/tis_data.py`. Both use the same module names as the original library, so **restart the kernel before running it**; it does not need part 1.

The paths of `sim_istarz_2603` (Langevin, γ = 2, positions only) are read with `tis_data.prepare_data`: every accepted path is cut at its first frame past its ensemble's interface M, its mirrored piece is added (`--reversed`), and every piece carries its path's rejection weight. The first 40% of the MC cycles of every ensemble are discarded (`--burn-in-fraction`), and `--paths-per-ensemble` keeps that many paths evenly spaced over the rest of the chain.

Positions only means no velocities in the data: `--fd-velocity` adds the change of y and x over one frame as two more columns (dy, dx; computed on the whole path before the cut) and they are the velocity CVs. The fit uses the lagged update (`toy2d.lag_update`, lag = 0.05 / (γ dt) = 12 frames): the dynamics is not Markov in (y, x, dy, dx) at lag 1, as the finite-difference velocity is a noisy velocity.

In [ ]:
import argparse
import os
import sys

import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

REPO = '/home/staff7/fbsskr/Notebooks/Students/elias/optimalrcs'     # partial_tis checkout
sys.path.insert(0, f'{REPO}/optimalrcs-cupy/')
sys.path.insert(0, f'{REPO}/examples/')
import optimalrcs, tis_data, toy2d

FIG = 'figures/nb_zpot_short_traj_langevin'
os.makedirs(FIG, exist_ok=True)

In [ ]:
TIS_DIR = '/home/staff7/fbsskr/Notebooks/Students/elias/data/simulations/sim_istarz_2603'
PATHS_PER_ENSEMBLE = 1500
DT, GAMMA_MD = 0.002, 2.0
FIT_LAG = max(1, int(round(0.05 / (GAMMA_MD * DT))))      # 12 frames

parser = argparse.ArgumentParser()
tis_data.add_data_arguments(parser)
args = parser.parse_args([TIS_DIR, '--reversed', '--weights', 'rejections', '--burn-in-fraction', '0.4',
                          '--paths-per-ensemble', str(PATHS_PER_ENSEMBLE), '--fd-velocity'])
data = tis_data.prepare_data(args)
y_path, x_path = data.X_all[:, 0], data.X_all[:, 1]
print(f'{len(y_path):,} frames in {len(np.unique(data.i_traj))} pieces from {len(np.unique(data.group_traj))} paths; '
      f'CVs y, x, dy, dx: {data.X.shape}')

In [ ]:
q_path = optimalrcs.CommittorNE(boundary0=data.boundary0, boundary1=data.boundary1,
                                i_traj=data.i_traj, t_traj=data.t_traj,
                                path_weights=data.path_weights,      # MC weight of each frame's path
                                group_traj=data.group_traj)          # forward and mirrored piece of a path: one unit
q_path.set_fixed_traj_length_trap(data.boundary0 | data.boundary1, 2 ** 16)

X_gpu = cp.asarray(data.X, dtype=cp.float64)                          # y, x, dy, dx
def comp_y_path():
    return X_gpu[:, np.random.randint(X_gpu.shape[1])]

extra_kw = toy2d.lag_update(optimalrcs, data.i_traj, data.boundary0 | data.boundary1, data.path_weights, FIT_LAG)
fit_kw = dict(ny=6, save_min_metric='max_z_zq_stopped',
              metrics_print=('iter', 'delta_r2', 'max_z_zq_stopped', 'max_sd_zq_stopped', 'delta_x', 'time_elapsed'),
              **extra_kw)
np.random.seed(0)
cp.random.seed(0)

optimalrcs.envelope_scale = 1           # warm-up
q_path.fit_transform(comp_y_path, gamma=0.1, max_iter=1000, print_step=500, **fit_kw)
optimalrcs.envelope_scale = 0.01        # main fit
q_path.fit_transform(comp_y_path, gamma=0.1, max_iter=15000, print_step=1000, **fit_kw)

q_path.plots_metrics(['delta_r2', 'max_z_zq_stopped', 'delta_x'])
q_path.plots_tis()                        # stopped Z_q: the test for path data
q_path.plots_feps(delta_t_sim=DT)         # with the path weights

In [ ]:
# save in the format part 1 reads (FIRST_TRY_NPZ)
OUT_NPZ = f'{FIG}/committor_staple_langevin_p{PATHS_PER_ENSEMBLE}.npz'
np.savez(OUT_NPZ, r=q_path.r_traj.get(), X_all=data.X_all, i_traj=data.i_traj, t_traj=data.t_traj,
         path_weights=data.path_weights, boundary0=data.boundary0, boundary1=data.boundary1)
print('saved', OUT_NPZ)